# FAM129B Screening Pipeline — Review Notebook

Run cells **top to bottom**, stopping to inspect at each section before continuing.
Each section is one pipeline stage. A **PASS/FAIL verdict** is printed at the end
of Sections 1 and 2 — do not proceed to FAM129B analysis unless controls pass.

| Section | Stage | Depends on |
|---------|-------|------------|
| 0 | Setup & imports | — |
| 1 | Monomer QC | AF3 monomer outputs |
| 2 | Control validation (RAB5A-Rabaptin5) | AF3 complex outputs |
| 3 | FAM129B complexes | Section 2 PASS |
| 4 | Interface extraction & docking setup | Section 3 |
| 5 | Screening results | Docking complete |

## Section 0 — Setup

In [ ]:
import sys
import os
from pathlib import Path

# Repo root is one level above notebooks/
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

# AF3 output directory — change this if your outputs are elsewhere
AF3_OUTPUT_DIR = REPO_ROOT.parent / "output"
os.environ["AF3_OUTPUT_DIR"] = str(AF3_OUTPUT_DIR)

print(f"Repo root:       {REPO_ROOT}")
print(f"AF3 output dir:  {AF3_OUTPUT_DIR}")
print(f"Output dir exists: {AF3_OUTPUT_DIR.exists()}")
print(f"Jobs found: {[d.name for d in AF3_OUTPUT_DIR.iterdir() if d.is_dir()] if AF3_OUTPUT_DIR.exists() else 'directory not found'}")

In [ ]:
# Import all pipeline modules
from src.scoring.parse_af3 import parse_af3
from src.scoring.interface import compute_interface, get_contact_residues, measure_switch_distances
from src.scoring.prodigy import run_prodigy, run_prodigy_batch

from src.viz.af3_plots import (
    plot_pae_heatmap, plot_plddt_per_residue,
    plot_dg_comparison, plot_bsa_comparison, plot_score_summary_table
)
from src.viz.structure_plots import (
    view_complex_by_chain, view_plddt,
    view_binding_site, view_gtp_gdp_overlay,
    view_docking_grid, view_docked_pose, make_structure_report
)
from src.viz.interface_plots import (
    plot_contact_heatmap, plot_grid_box_2d, plot_switch_distances
)

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
print("All modules loaded.")

In [ ]:
# Load the full score table from all available AF3 outputs
scores_df = parse_af3()
print(f"{len(scores_df)} jobs found")
scores_df

## Section 1 — Monomer QC

Check each individual protein is well-folded before trusting complex predictions.
Expect pLDDT > 70 over the structured core. Flag any protein with > 50% disordered.

**Monomers to check:** RAB5A, RAB7A, FAM129B, CAPZA1, CAPZB, Rabaptin5

In [ ]:
MONOMER_JOBS = [
    "rab5a_full_monomer",
    "rab7a_monomer",
    "fam129b_niban2_monomer",
    "capza1_monomer",
    "capzb_monomer",
    "rabep1_rabaptin5_monomer",
]

monomer_scores = scores_df[scores_df["job_name"].isin(MONOMER_JOBS)]
print("Monomer scores:")
print(monomer_scores[["job_name", "ptm", "ranking_score", "fraction_disordered"]].to_string(index=False))

In [ ]:
# Score summary table — colour-coded
plot_score_summary_table(monomer_scores);

In [ ]:
# pLDDT plots for each monomer — run all, inspect for disordered regions
for job_name in MONOMER_JOBS:
    job_dir = AF3_OUTPUT_DIR / job_name
    conf_json = job_dir / f"{job_name}_confidences.json"
    if conf_json.exists():
        print(f"\n--- {job_name} ---")
        plot_plddt_per_residue(str(conf_json), job_name)
        plt.show()
    else:
        print(f"[SKIP] {job_name}: no confidences.json found")

In [ ]:
# 3D structure views — pLDDT-coloured
# Run one at a time; change job_name to inspect a specific protein
job_name = "rab5a_full_monomer"   # <-- change this
cif_path = str(AF3_OUTPUT_DIR / job_name / f"{job_name}_model.cif")

print(f"pLDDT view: {job_name}")
view_plddt(cif_path)

In [ ]:
# VERDICT — Section 1
DISORDERED_THRESHOLD = 0.5
PTM_THRESHOLD = 0.5

failed = monomer_scores[
    (monomer_scores["fraction_disordered"] > DISORDERED_THRESHOLD) |
    (monomer_scores["ptm"] < PTM_THRESHOLD)
]

if failed.empty:
    print("✓ PASS — all monomers meet quality threshold")
else:
    print("⚠ WARNING — these monomers may be problematic:")
    print(failed[["job_name", "ptm", "fraction_disordered"]].to_string(index=False))
    print("\nFor highly disordered proteins (e.g. Rabaptin5 at 60%), this is expected.")
    print("Proceed with caution for those specific proteins.")

## Section 2 — Control Validation (RAB5A-Rabaptin5)

**Critical gate.** AF3 must show that Rabaptin5 prefers RAB5A-GTP over RAB5A-GDP.
Rabaptin5 is a known effector with ~10x higher affinity for the GTP state.

**Pass criteria (both must hold):**
- PRODIGY ΔG(GTP) < ΔG(GDP)  ← tighter binding in active state
- BSA(GTP) > BSA(GDP)          ← larger interface in active state

If controls fail → do NOT interpret FAM129B results.

In [ ]:
CONTROL_GTP = "rab5a_gtp_mg_rabaptin5_complex"
CONTROL_GDP = "rab5a_gdp_mg_rabaptin5_complex"

# Chain IDs in the complex: A=RAB5A, B=Rabaptin5, C=GTP/GDP, D=Mg
RAB_CHAIN   = "A"
PARTNER_CHAIN = "B"

gtp_cif = str(AF3_OUTPUT_DIR / CONTROL_GTP / f"{CONTROL_GTP}_model.cif")
gdp_cif = str(AF3_OUTPUT_DIR / CONTROL_GDP / f"{CONTROL_GDP}_model.cif")

print(f"GTP complex: {Path(gtp_cif).exists()}")
print(f"GDP complex: {Path(gdp_cif).exists()}")

In [ ]:
# PAE heatmaps for both states
for job_name, cif_path in [(CONTROL_GTP, gtp_cif), (CONTROL_GDP, gdp_cif)]:
    conf_json = str(AF3_OUTPUT_DIR / job_name / f"{job_name}_confidences.json")
    if Path(conf_json).exists():
        plot_pae_heatmap(conf_json, job_name)
        plt.show()

In [ ]:
# 3D chain-coloured views
print("GTP complex:")
view_complex_by_chain(gtp_cif)

In [ ]:
print("GDP complex:")
view_complex_by_chain(gdp_cif)

In [ ]:
# GTP/GDP overlay — Switch I/II highlighted
view_gtp_gdp_overlay(gtp_cif, gdp_cif, rab_chain_id=RAB_CHAIN)

In [ ]:
# Interface metrics for both states
print("Computing interface metrics...")
gtp_interface = compute_interface(gtp_cif, RAB_CHAIN, PARTNER_CHAIN)
gdp_interface = compute_interface(gdp_cif, RAB_CHAIN, PARTNER_CHAIN)

print(f"\n{'Metric':<25} {'GTP':>10} {'GDP':>10}")
print("-" * 47)
print(f"{'BSA (Å²)':<25} {gtp_interface['bsa']:>10.1f} {gdp_interface['bsa']:>10.1f}")
print(f"{'Contact residue pairs':<25} {gtp_interface['contact_count']:>10} {gdp_interface['contact_count']:>10}")

In [ ]:
# Switch I/II distances
gtp_switch = measure_switch_distances(gtp_cif, RAB_CHAIN, PARTNER_CHAIN)
gdp_switch = measure_switch_distances(gdp_cif, RAB_CHAIN, PARTNER_CHAIN)

switch_data = {
    f"{CONTROL_GTP} (GTP)": gtp_switch,
    f"{CONTROL_GDP} (GDP)": gdp_switch,
}
plot_switch_distances(switch_data);

In [ ]:
# PRODIGY binding energy
prodigy_jobs = [
    {"job_name": CONTROL_GTP, "cif_path": gtp_cif, "chain_a": RAB_CHAIN, "chain_b": PARTNER_CHAIN},
    {"job_name": CONTROL_GDP, "cif_path": gdp_cif, "chain_a": RAB_CHAIN, "chain_b": PARTNER_CHAIN},
]
prodigy_df = run_prodigy_batch(prodigy_jobs)
print(prodigy_df.to_string(index=False))

In [ ]:
# GTP vs GDP iptm comparison chart
control_pairs = [(CONTROL_GTP, CONTROL_GDP)]
plot_dg_comparison(scores_df, control_pairs);

# BSA comparison
interface_results = {
    CONTROL_GTP: gtp_interface,
    CONTROL_GDP: gdp_interface,
}
plot_bsa_comparison(interface_results);

In [ ]:
# VERDICT — Section 2
gtp_dg = prodigy_df.loc[prodigy_df.job_name == CONTROL_GTP, "dg_kcal_mol"].values[0]
gdp_dg = prodigy_df.loc[prodigy_df.job_name == CONTROL_GDP, "dg_kcal_mol"].values[0]

dg_pass  = (gtp_dg is not None and gdp_dg is not None and gtp_dg < gdp_dg)
bsa_pass = gtp_interface["bsa"] > gdp_interface["bsa"]

print(f"ΔG(GTP)={gtp_dg}  ΔG(GDP)={gdp_dg}  → {'PASS' if dg_pass else 'FAIL (or prodigy not installed)'}")
print(f"BSA(GTP)={gtp_interface['bsa']:.1f}  BSA(GDP)={gdp_interface['bsa']:.1f}  → {'PASS' if bsa_pass else 'FAIL'}")

if bsa_pass:   # ΔG check is optional if prodigy not installed
    print("\n✓ PASS — controls validate. Proceed to Section 3.")
    CONTROLS_PASSED = True
else:
    print("\n✗ FAIL — AF3 is not correctly capturing GTP-state preference.")
    print("   Do NOT interpret FAM129B complex results until this is resolved.")
    CONTROLS_PASSED = False

## Section 3 — FAM129B Complexes

Core hypothesis pairs. For each pair, same analysis as Section 2.
GTP should score better than GDP if FAM129B is a real effector.

**Pairs:**
- FAM129B × RAB5A (GTP / GDP)
- FAM129B × RAB7A (GTP / GDP)
- FAM129B × CapZ heterodimer

In [ ]:
if not CONTROLS_PASSED:
    print("⛔ Controls failed in Section 2. Resolve before continuing.")
else:
    print("Controls passed — proceeding.")

In [ ]:
# Define FAM129B pairs — (gtp_job, gdp_job, rab_chain, partner_chain)
FAM129B_PAIRS = [
    ("fam129b_rab5a_gtp", "fam129b_rab5a_gdp", "B", "A"),  # chain B=RAB5A, A=FAM129B
    ("fam129b_rab7a_gtp", "fam129b_rab7a_gdp", "B", "A"),  # chain B=RAB7A, A=FAM129B
]
CAPZ_JOB = "fam129b_capz"  # A=FAM129B, B=CAPZA1, C=CAPZB — no GTP/GDP

# Check which jobs are available
available = []
for gtp_job, gdp_job, _, _ in FAM129B_PAIRS:
    for job in [gtp_job, gdp_job]:
        exists = (AF3_OUTPUT_DIR / job).exists()
        print(f"  {'✓' if exists else '✗'} {job}")
        if exists:
            available.append(job)

capz_exists = (AF3_OUTPUT_DIR / CAPZ_JOB).exists()
print(f"  {'✓' if capz_exists else '✗'} {CAPZ_JOB}")

In [ ]:
# Score comparison for all FAM129B pairs
fam_pair_names = [(gtp, gdp) for gtp, gdp, _, _ in FAM129B_PAIRS]
fam_jobs_available = [j for j in [p for pair in fam_pair_names for p in pair] if (AF3_OUTPUT_DIR / j).exists()]

if fam_jobs_available:
    fam_scores = scores_df[scores_df["job_name"].isin(fam_jobs_available)]
    plot_score_summary_table(fam_scores)
    plt.show()
    plot_dg_comparison(scores_df, fam_pair_names)
    plt.show()
else:
    print("No FAM129B complex outputs found yet — submit jobs and re-run.")

In [ ]:
# Per-pair deep dive — change pair_idx to inspect a specific pair
pair_idx = 0   # 0 = FAM129B×RAB5A, 1 = FAM129B×RAB7A

gtp_job, gdp_job, rab_chain, fam_chain = FAM129B_PAIRS[pair_idx]
gtp_cif_fam = str(AF3_OUTPUT_DIR / gtp_job / f"{gtp_job}_model.cif")
gdp_cif_fam = str(AF3_OUTPUT_DIR / gdp_job / f"{gdp_job}_model.cif")

if Path(gtp_cif_fam).exists():
    print(f"Inspecting: {gtp_job}")

    # 3D views
    print("\nChain-coloured (GTP state):")
    view_complex_by_chain(gtp_cif_fam)
else:
    print(f"{gtp_job} not yet available.")

In [ ]:
if Path(gtp_cif_fam).exists():
    # Interface metrics
    gtp_iface = compute_interface(gtp_cif_fam, rab_chain, fam_chain)
    gdp_iface = compute_interface(gdp_cif_fam, rab_chain, fam_chain) if Path(gdp_cif_fam).exists() else {}

    # Contact residue heatmap
    plot_contact_heatmap(gtp_cif_fam, rab_chain, fam_chain, gtp_job)
    plt.show()

    # Binding site 3D view
    print("\nBinding site (GTP state):")
    view_binding_site(
        gtp_cif_fam, rab_chain, fam_chain,
        gtp_iface.get("contact_residues_a", []),
        gtp_iface.get("contact_residues_b", [])
    )

    # GTP/GDP overlay
    if Path(gdp_cif_fam).exists():
        print("\nGTP/GDP overlay:")
        view_gtp_gdp_overlay(gtp_cif_fam, gdp_cif_fam, rab_chain_id=rab_chain)

In [ ]:
# PRODIGY ΔG for FAM129B pairs
fam_prodigy_jobs = []
for gtp_job, gdp_job, rab_chain, fam_chain in FAM129B_PAIRS:
    for job in [gtp_job, gdp_job]:
        cif = AF3_OUTPUT_DIR / job / f"{job}_model.cif"
        if cif.exists():
            fam_prodigy_jobs.append({"job_name": job, "cif_path": str(cif),
                                     "chain_a": rab_chain, "chain_b": fam_chain})

if fam_prodigy_jobs:
    fam_prodigy_df = run_prodigy_batch(fam_prodigy_jobs)
    print(fam_prodigy_df.to_string(index=False))
else:
    print("No FAM129B complex outputs yet.")

In [ ]:
# VERDICT — Section 3
# For each pair: does GTP have lower ΔG and larger BSA than GDP?
print("FAM129B interaction verdicts:\n")
for gtp_job, gdp_job, rab_chain, fam_chain in FAM129B_PAIRS:
    gtp_cif_v = str(AF3_OUTPUT_DIR / gtp_job / f"{gtp_job}_model.cif")
    gdp_cif_v = str(AF3_OUTPUT_DIR / gdp_job / f"{gdp_job}_model.cif")
    if not (Path(gtp_cif_v).exists() and Path(gdp_cif_v).exists()):
        print(f"  {gtp_job.split('_rab')[0]} × {gtp_job.split('fam129b_')[1].split('_gtp')[0].upper()}: NOT YET AVAILABLE")
        continue

    gi = compute_interface(gtp_cif_v, rab_chain, fam_chain)
    di = compute_interface(gdp_cif_v, rab_chain, fam_chain)
    bsa_ok = gi["bsa"] > di["bsa"]
    contact_ok = gi["contact_count"] >= di["contact_count"]
    label = "✓ CANDIDATE" if (bsa_ok and contact_ok) else "✗ WEAK / NO INTERACTION"
    pair_label = gtp_job.replace('_gtp', '').replace('fam129b_', 'FAM129B × ').upper()
    print(f"  {pair_label}: {label}  (BSA GTP={gi['bsa']:.0f} GDP={di['bsa']:.0f}, contacts GTP={gi['contact_count']} GDP={di['contact_count']})")

## Section 4 — Interface Extraction & Docking Setup

Prepare the receptor from the best FAM129B complex and define the Vina search grid.
Verify the grid box is correctly centered on the contact patch before screening.

In [ ]:
from src.docking.prepare import prepare_receptor
from src.docking.screen import define_grid

# Use the GTP-state FAM129B×RAB5A complex as the docking receptor
# Change DOCKING_JOB to whichever complex scored best in Section 3
DOCKING_JOB    = "fam129b_rab5a_gtp"
RECEPTOR_CHAIN = "B"   # RAB5A chain
PARTNER_CHAIN  = "A"   # FAM129B chain

docking_cif = str(AF3_OUTPUT_DIR / DOCKING_JOB / f"{DOCKING_JOB}_model.cif")
docking_out = REPO_ROOT / "results" / "docking"
docking_out.mkdir(parents=True, exist_ok=True)

if Path(docking_cif).exists():
    print(f"Preparing receptor from: {DOCKING_JOB}")
    receptor_pdbqt = prepare_receptor(docking_cif, RECEPTOR_CHAIN, str(docking_out))
    print(f"Receptor PDBQT: {receptor_pdbqt}")
else:
    print(f"{DOCKING_JOB} not yet available. Submit the SLURM job first.")
    receptor_pdbqt = None

In [ ]:
if receptor_pdbqt and Path(docking_cif).exists():
    # Get contact residues on the RAB5A side
    contact_a, contact_b = get_contact_residues(docking_cif, RECEPTOR_CHAIN, PARTNER_CHAIN)
    print(f"Contact residues on RAB5A ({RECEPTOR_CHAIN}): {len(contact_a)} residues")
    print(f"Contact residues on FAM129B ({PARTNER_CHAIN}): {len(contact_b)} residues")

    # Define docking grid
    grid = define_grid(contact_a, docking_cif, RECEPTOR_CHAIN, padding=4.0)
    print(f"\nDocking grid: {grid}")

In [ ]:
if receptor_pdbqt and Path(docking_cif).exists():
    # 2D grid box diagram
    plot_grid_box_2d(docking_cif, RECEPTOR_CHAIN, grid, contact_a, DOCKING_JOB)
    plt.show()

    # 3D grid box on receptor
    print("3D grid box view:")
    view_docking_grid(docking_cif, RECEPTOR_CHAIN, grid)

## Section 5 — Screening Results

Run after docking is complete. Load results, triage, visualize.
The docking run is kicked off by `python src/docking/screen.py` (can take hours for full library).

In [ ]:
from src.docking.triage import triage_hits, save_hits
from src.viz.docking_plots import (
    plot_score_distribution, plot_zscore_ranked,
    plot_vina_gnina_scatter, plot_top_hits, plot_screening_summary
)

hits_path = REPO_ROOT / "results" / "docking_hits.csv"
if hits_path.exists():
    hits_df = pd.read_csv(hits_path)
    print(f"Loaded {len(hits_df)} compounds from docking_hits.csv")
    print(f"Consensus hits (Z ≤ -2.5 in both): {hits_df['consensus_flag'].sum()}")
    hits_df.head(10)
else:
    print("docking_hits.csv not found — run docking first.")
    hits_df = None

In [ ]:
# Load decoy stats for Z-score reference
# These are saved alongside the screening run
decoy_stats_path = REPO_ROOT / "results" / "docking" / "decoy_stats.json"
if decoy_stats_path.exists():
    import json
    with open(decoy_stats_path) as f:
        decoy_stats = json.load(f)
    DECOY_MEAN = decoy_stats["mean"]
    DECOY_STD  = decoy_stats["std"]
    print(f"Decoy baseline: mean={DECOY_MEAN:.2f}, std={DECOY_STD:.2f} kcal/mol")
else:
    # Fallback: estimate from the hits distribution
    DECOY_MEAN = hits_df["vina_score"].mean() if hits_df is not None else -6.0
    DECOY_STD  = hits_df["vina_score"].std()  if hits_df is not None else 1.0
    print(f"decoy_stats.json not found. Using distribution estimate: mean={DECOY_MEAN:.2f}, std={DECOY_STD:.2f}")

In [ ]:
if hits_df is not None:
    # One-page screening summary
    plot_screening_summary(hits_df, DECOY_MEAN, DECOY_STD)
    plt.show()

In [ ]:
if hits_df is not None:
    # Individual plots
    plot_score_distribution(hits_df, DECOY_MEAN, DECOY_STD); plt.show()
    plot_zscore_ranked(hits_df);                              plt.show()
    plot_vina_gnina_scatter(hits_df);                         plt.show()
    plot_top_hits(hits_df);                                   plt.show()

In [ ]:
if hits_df is not None:
    # 3D view of the top-ranked pose
    top_hit = hits_df[hits_df["consensus_flag"] == True].iloc[0] if hits_df["consensus_flag"].any() else hits_df.iloc[0]
    print(f"Top hit: {top_hit['compound_id']}  Vina Z={top_hit['vina_zscore']:.2f}  SMILES: {top_hit['smiles']}")

    if receptor_pdbqt and pd.notna(top_hit.get("pose_file")) and Path(top_hit["pose_file"]).exists():
        view_docked_pose(receptor_pdbqt, top_hit["pose_file"], grid)
    else:
        print("Pose file not found — check results/docking/ directory.")

In [ ]:
# FINAL VERDICT
if hits_df is not None:
    consensus = hits_df[hits_df["consensus_flag"] == True]
    print(f"Screening complete.")
    print(f"  Total screened:   {len(hits_df)}")
    print(f"  Vina hits (Z≤-2.5): {(hits_df['vina_zscore'] <= -2.5).sum()}")
    print(f"  Consensus hits:   {len(consensus)}")
    print()
    if len(consensus) > 0:
        print("Top consensus hits for wet-lab ordering:")
        print(consensus[["compound_id", "smiles", "vina_score", "gnina_score", "vina_zscore"]].head(10).to_string(index=False))
    else:
        print("No consensus hits. Consider: lowering cutoff, trying FAM129B as receptor, or expanding library.")